Bayesian Network for Heart Disease Diagnosis
Aim
To construct a Bayesian Network considering medical data and demonstrate the diagnosis of heart patients using the standard Heart Disease Dataset.

Dataset
This notebook uses the uploaded Heart.csv dataset.

Target variable: AHD

Yes → Heart disease present
No → Heart disease absent
Libraries
pandas
numpy
scikit-learn
pgmpy

In [ ]:
!pip install -q pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 15.9 MB/s eta 0:00:00


In [ ]:
# Install pgmpy if it is not already installed
# Run this cell if required:
# %pip install pgmpy pandas numpy scikit-learn
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

print("Libraries imported successfully.")

Libraries imported successfully.


/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [13]:
# Load the dataset/
file_path = "/content/Heart.csv"
data = pd.read_csv("/content/Heart.csv")

print("Dataset shape:", data.shape)
display(data.head())


Dataset shape: (303, 15)


,Unnamed: 0,Age,Sex,ChestPain,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,AHD
0,1,63,1,typical,145,233,1,2,150,0,2.3,3,0.0,fixed,No
1,2,67,1,asymptomatic,160,286,0,2,108,1,1.5,2,3.0,normal,Yes
2,3,67,1,asymptomatic,120,229,0,2,129,1,2.6,2,2.0,reversable,Yes
3,4,37,1,nonanginal,130,250,0,0,187,0,3.5,3,0.0,normal,No
4,5,41,0,nontypical,130,204,0,2,172,0,1.4,1,0.0,normal,No


In [14]:
# Display dataset information
print("Column names:")
print(data.columns.tolist())

print("\nMissing values:")
print(data.isnull().sum())

print("\nTarget distribution:")
print(data["AHD"].value_counts())

Column names:
['Unnamed: 0', 'Age', 'Sex', 'ChestPain', 'RestBP', 'Chol', 'Fbs', 'RestECG', 'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'AHD']

Missing values:
Unnamed: 0    0
Age           0
Sex           0
ChestPain     0
RestBP        0
Chol          0
Fbs           0
RestECG       0
MaxHR         0
ExAng         0
Oldpeak       0
Slope         0
Ca            4
Thal          2
AHD           0
dtype: int64

Target distribution:
AHD
No     164
Yes    139
Name: count, dtype: int64


In [15]:
# Remove the index column if present
if "Unnamed: 0" in data.columns:
    data = data.drop(columns=["Unnamed: 0"])

# Handle missing values
if "Ca" in data.columns:
    data["Ca"] = data["Ca"].fillna(data["Ca"].mode()[0])

if "Thal" in data.columns:
    data["Thal"] = data["Thal"].fillna(data["Thal"].mode()[0])

print("Missing values after treatment:")
print(data.isnull().sum())

Missing values after treatment:
Age          0
Sex          0
ChestPain    0
RestBP       0
Chol         0
Fbs          0
RestECG      0
MaxHR        0
ExAng        0
Oldpeak      0
Slope        0
Ca           0
Thal         0
AHD          0
dtype: int64


## Data Discretization

A discrete Bayesian Network is used in this practical. Therefore, continuous variables are converted into meaningful categorical groups.

### Age
- 0–40 → Young
- 41–50 → Middle
- 51–60 → Senior
- 61+ → Old

### Maximum Heart Rate
- 0–100 → Low
- 101–130 → Moderate
- 131–160 → High
- 161+ → VeryHigh

In [16]:
# Create categorical variables

data["AgeGroup"] = pd.cut(
    data["Age"],
    bins=[0, 40, 50, 60, 100],
    labels=["Young", "Middle", "Senior", "Old"]
).astype(str)

data["MaxHRGroup"] = pd.cut(
    data["MaxHR"],
    bins=[0, 100, 130, 160, 250],
    labels=["Low", "Moderate", "High", "VeryHigh"]
).astype(str)

# Convert categorical variables to strings
data["ChestPain"] = data["ChestPain"].astype(str)
data["ExAng"] = data["ExAng"].astype(str)
data["Ca"] = data["Ca"].astype(int).astype(str)
data["Thal"] = data["Thal"].astype(str)
data["AHD"] = data["AHD"].astype(str)

display(data[[
    "AgeGroup", "ChestPain", "MaxHRGroup",
    "ExAng", "Ca", "Thal", "AHD"
]].head())

,AgeGroup,ChestPain,MaxHRGroup,ExAng,Ca,Thal,AHD
0,Old,typical,High,0,0,fixed,No
1,Old,asymptomatic,Moderate,1,3,normal,Yes
2,Old,asymptomatic,Moderate,1,2,reversable,Yes
3,Young,nonanginal,VeryHigh,0,0,normal,No
4,Middle,nontypical,VeryHigh,0,0,normal,No


In [17]:
# Select variables for the Bayesian Network
bn_data = data[
    [
        "AgeGroup",
        "ChestPain",
        "MaxHRGroup",
        "ExAng",
        "Ca",
        "Thal",
        "AHD"
    ]
].copy()

print("Bayesian Network data shape:", bn_data.shape)
display(bn_data.head())


Bayesian Network data shape: (303, 7)


,AgeGroup,ChestPain,MaxHRGroup,ExAng,Ca,Thal,AHD
0,Old,typical,High,0,0,fixed,No
1,Old,asymptomatic,Moderate,1,3,normal,Yes
2,Old,asymptomatic,Moderate,1,2,reversable,Yes
3,Young,nonanginal,VeryHigh,0,0,normal,No
4,Middle,nontypical,VeryHigh,0,0,normal,No


In [18]:
# Split into training and testing datasets
train_data, test_data = train_test_split(
    bn_data,
    test_size=0.20,
    random_state=42,
    stratify=bn_data["AHD"]
)

print("Training records:", len(train_data))
print("Testing records:", len(test_data))

Training records: 242
Testing records: 61


## Construct the Bayesian Network

The network structure is:

**AgeGroup → AHD**  
**ChestPain → AHD**  
**MaxHRGroup → AHD**  
**ExAng → AHD**  
**Ca → AHD**  
**Thal → AHD**

Thus, the model estimates:

\[
P(AHD \mid AgeGroup, ChestPain, MaxHRGroup, ExAng, Ca, Thal)
\]


In [19]:
# Construct Bayesian Network
model = DiscreteBayesianNetwork([
    ("AgeGroup", "AHD"),
    ("ChestPain", "AHD"),
    ("MaxHRGroup", "AHD"),
    ("ExAng", "AHD"),
    ("Ca", "AHD"),
    ("Thal", "AHD")
])

print("Bayesian Network structure:")
print(list(model.edges()))

Bayesian Network structure:
[('AgeGroup', 'AHD'), ('ChestPain', 'AHD'), ('MaxHRGroup', 'AHD'), ('ExAng', 'AHD'), ('Ca', 'AHD'), ('Thal', 'AHD')]


In [29]:
# Learn the Conditional Probability Tables (CPTs)

estimator = BayesianEstimator(model, train_data)

cpds = estimator.get_parameters(
    prior_type="BDeu",
    equivalent_sample_size=10
)

model.add_cpds(*cpds)

print("Model valid:", model.check_model())

/tmp/ipykernel_1224/4184423778.py:3: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(model, train_data)


Model valid: True


In [30]:
# Display all learned CPTs
print("===== CONDITIONAL PROBABILITY TABLES =====")

for cpd in model.get_cpds():
    print("\n", cpd)


===== CONDITIONAL PROBABILITY TABLES =====

 +------------------+-----------+
| AgeGroup(Middle) | 0.259921  |
+------------------+-----------+
| AgeGroup(Old)    | 0.263889  |
+------------------+-----------+
| AgeGroup(Senior) | 0.414683  |
+------------------+-----------+
| AgeGroup(Young)  | 0.0615079 |
+------------------+-----------+

 +------------+-----+----------------------+
| AgeGroup   | ... | AgeGroup(Young)      |
+------------+-----+----------------------+
| Ca         | ... | Ca(3)                |
+------------+-----+----------------------+
| ChestPain  | ... | ChestPain(typical)   |
+------------+-----+----------------------+
| ExAng      | ... | ExAng(1)             |
+------------+-----+----------------------+
| MaxHRGroup | ... | MaxHRGroup(VeryHigh) |
+------------+-----+----------------------+
| Thal       | ... | Thal(reversable)     |
+------------+-----+----------------------+
| AHD(No)    | ... | 0.5                  |
+------------+-----+--------------------

In [31]:
# Create inference engine
inference = VariableElimination(model)
print("Inference engine created successfully.")

Inference engine created successfully.


## Diagnosis of a Sample Patient

The following hypothetical patient's medical information is supplied as evidence to the Bayesian Network.

- Age group: Senior
- Chest pain: asymptomatic
- Maximum heart rate: Moderate
- Exercise-induced angina: 1
- Number of major vessels: 1
- Thalassemia: reversable


In [32]:
# Evidence for a sample patient
patient = {
    "AgeGroup": "Senior",
    "ChestPain": "asymptomatic",
    "MaxHRGroup": "Moderate",
    "ExAng": "1",
    "Ca": "1",
    "Thal": "reversable"
}

result = inference.query(
    variables=["AHD"],
    evidence=patient
)

print("===== PATIENT DIAGNOSIS =====")
print(result)

===== PATIENT DIAGNOSIS =====
+----------+------------+
| AHD      |   phi(AHD) |
+==========+============+
| AHD(No)  |     0.0016 |
+----------+------------+
| AHD(Yes) |     0.9984 |
+----------+------------+


In [33]:
# Extract posterior probabilities
state_names = result.state_names["AHD"]
probabilities = dict(zip(state_names, result.values))

probability_yes = probabilities.get("Yes", 0)
probability_no = probabilities.get("No", 0)

print(f"Probability of Heart Disease (Yes): {probability_yes:.4f}")
print(f"Probability of No Heart Disease (No): {probability_no:.4f}")

if probability_yes > probability_no:
    print("\nDiagnosis: HEART DISEASE PRESENT")
else:
    print("\nDiagnosis: NO HEART DISEASE")

Probability of Heart Disease (Yes): 0.9984
Probability of No Heart Disease (No): 0.0016

Diagnosis: HEART DISEASE PRESENT


## Model Evaluation

The trained Bayesian Network is evaluated on the test set. For every patient, the observed medical variables are supplied as evidence and the class with the highest posterior probability is selected as the prediction.


In [34]:
# Evaluate Bayesian Network on test data

predictions = []
actual = []

feature_columns = [
    "AgeGroup",
    "ChestPain",
    "MaxHRGroup",
    "ExAng",
    "Ca",
    "Thal"
]

for _, row in test_data.iterrows():

    evidence = {column: row[column] for column in feature_columns}

    result = inference.query(
        variables=["AHD"],
        evidence=evidence
    )

    predicted_class = result.state_names["AHD"][
        np.argmax(result.values)
    ]

    predictions.append(predicted_class)
    actual.append(row["AHD"])

accuracy = accuracy_score(actual, predictions)

print("===== MODEL PERFORMANCE =====")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy (%): {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(actual, predictions, zero_division=0))


===== MODEL PERFORMANCE =====
Accuracy: 0.6557
Accuracy (%): 65.57%

Classification Report:
              precision    recall  f1-score   support

          No       0.62      0.94      0.75        33
         Yes       0.82      0.32      0.46        28

    accuracy                           0.66        61
   macro avg       0.72      0.63      0.60        61
weighted avg       0.71      0.66      0.62        61



## Optional: Diagnose an Actual Patient from the Test Dataset

The next cell selects one actual patient from the dataset and uses the remaining medical attributes as evidence.


In [35]:
# Diagnose one actual patient from the test set

sample = test_data.iloc[0]

evidence = {column: sample[column] for column in feature_columns}

result = inference.query(
    variables=["AHD"],
    evidence=evidence
)

predicted_class = result.state_names["AHD"][
    np.argmax(result.values)
]

print("Patient evidence:")
for key, value in evidence.items():
    print(f"{key}: {value}")

print("\nActual diagnosis:", sample["AHD"])
print("Predicted diagnosis:", predicted_class)

print("\nPosterior probabilities:")
print(result)

Patient evidence:
AgeGroup: Senior
ChestPain: asymptomatic
MaxHRGroup: VeryHigh
ExAng: 0
Ca: 0
Thal: normal

Actual diagnosis: No
Predicted diagnosis: Yes

Posterior probabilities:
+----------+------------+
| AHD      |   phi(AHD) |
+==========+============+
| AHD(No)  |     0.0032 |
+----------+------------+
| AHD(Yes) |     0.9968 |
+----------+------------+


# Result

A Bayesian Network was successfully constructed using the Heart Disease Dataset. Conditional Probability Tables were learned from the training data using Bayesian estimation. Bayesian inference was then used to calculate the posterior probability of heart disease for a patient based on observed medical evidence.

The model was also evaluated on unseen test data using classification accuracy and a classification report.

**Note:** The exact probability and accuracy values are generated when the notebook is executed and should not be manually hard-coded.
